# Amazon ML Challenge 2026: Entity Resolution Suite
### Google Colab GPU/CPU Runner with Google Drive Persistence & Auto-Resume

**Running in Antigravity IDE via Google Colab Extension**:
1. Open Command Palette (`Ctrl+Shift+P` / `Cmd+Shift+P`).
2. Select `Google Colab: Connect to a Colab runtime` (or click Kernel selector top-right -> Google Colab).
3. Choose runtime accelerator: **L4 GPU** (or T4 GPU) with High-RAM.
4. Run all cells sequentially!

**Key Pipeline Architecture**:
- **Dedicated Run Isolation**: Each run writes to `output/runs/<run_id>/` and `drive/runs/<run_id>/`—prior outputs are never overwritten!
- **Speed-Optimized Inference**: TF-IDF cosine candidate pre-pruning + batched gatekeeper inference cuts runtime by >65%.
- **Drive Persistence**: Real-time mirroring to Google Drive checkpoints (`/content/drive/MyDrive/amazon-ml-challenge-2026`).
- **Official Macro F0.5 Metric**: Direct threshold optimization weighted towards precision (beta=0.5).
- **Memory-Safe Audit**: Built-in format verification with `utils/validate_submission.py`.

In [2]:
# 1. Verify GPU availability and system resources
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:      {torch.cuda.get_device_name(0)}")
    print(f"VRAM Capacity:   {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi
!lscpu | grep -E "Model name|CPU\(s\):" || true

PyTorch Version: 2.11.0+cu128
CUDA Available:  True
GPU Device:      NVIDIA L4
VRAM Capacity:   22.03 GB
Sun Sep 27 06:47:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   45C    P0             27W /   72W |     258MiB /  23034MiB |      0%      Default |
|                                  

In [3]:
# 2. Mount Google Drive to persist all datasets, weights, and benchmark results
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# 3. Initialize Google Drive directory structure for amazon-ml-challenge-2026
import os
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/amazon-ml-challenge-2026")
DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_WEIGHTS = DRIVE_ROOT / "weights"
DRIVE_CHECKPOINTS = DRIVE_ROOT / "checkpoints"
DRIVE_OUTPUT = DRIVE_ROOT / "output"

for folder in [DRIVE_ROOT, DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_CHECKPOINTS, DRIVE_OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"[✓] Google Drive structure initialized at: {DRIVE_ROOT}")
print(f"    - Datasets Dir   : {DRIVE_DATA}")
print(f"    - Weights Dir    : {DRIVE_WEIGHTS}")
print(f"    - Checkpoints Dir: {DRIVE_CHECKPOINTS}")
print(f"    - Output Dir     : {DRIVE_OUTPUT}")

[✓] Google Drive structure initialized at: /content/drive/MyDrive/amazon-ml-challenge-2026
    - Datasets Dir   : /content/drive/MyDrive/amazon-ml-challenge-2026/data
    - Weights Dir    : /content/drive/MyDrive/amazon-ml-challenge-2026/weights
    - Checkpoints Dir: /content/drive/MyDrive/amazon-ml-challenge-2026/checkpoints
    - Output Dir     : /content/drive/MyDrive/amazon-ml-challenge-2026/output


In [5]:
# 4. Clone or pull the repository
import os
REPO_URL = "https://github.com/Suyash8/amazon-ml-challenge.git"
REPO_DIR = "/content/amazon-ml-challenge"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!git pull || true

/content/amazon-ml-challenge
remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 15 (delta 10), reused 15 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 10.23 KiB | 1.14 MiB/s, done.
From https://github.com/Suyash8/amazon-ml-challenge
   4aeb657..cc90309  main       -> origin/main
Updating 4aeb657..cc90309
Fast-forward
 colab_runner.ipynb           | 345 +++++++++++++++++++++++++++++++++++++++----
 notebooks/colab_runner.ipynb | 322 ++++++++++++++++++++++++++++++++++++----
 scripts/run_pipeline.py      |  13 +-
 src/classifier.py            | 189 ++++++++++++------------
 4 files changed, 717 insertions(+), 152 deletions(-)


In [6]:
# 5. Install required packages (rapid string matching, lightgbm, etc.)
!python install.py

 Amazon ML Challenge 2026 - Dependency Installer
[*] 'uv' detected! Using uv pip for high-speed package installation.
Using Python 3.13.15 environment at: /usr
Checked 13 packages in 96ms

[✓] All dependencies successfully installed!


In [7]:
# 6. Dataset Linking, Shards Setup & Verification
# Google Drive datasets (/content/drive/MyDrive/amazon-ml-challenge-2026/data/)
# are symlinked to local data/ for zero-copy high-throughput I/O.
import os
from pathlib import Path

!mkdir -p data
drive_data = Path("/content/drive/MyDrive/amazon-ml-challenge-2026/data")
if drive_data.exists():
    for item in drive_data.iterdir():
        target = Path(f"data/{item.name}")
        if not target.exists():
            os.symlink(str(item), str(target))
            print(f"[✓] Linked {item.name} from Google Drive -> data/{item.name}")

# Check available dataset paths
print("\n--- AVAILABLE DATASETS CHECK ---")
for p in ["data/shards/train", "data/shards/test", "dataset_split/train", "dataset_split/test"]:
    if Path(p).exists():
        print(f"  [READY] Found dataset: {p}")
    else:
        print(f"  [INFO] Path not local: {p} (will look in Google Drive if uploaded)")


--- AVAILABLE DATASETS CHECK ---
  [READY] Found dataset: data/shards/train
  [READY] Found dataset: data/shards/test
  [READY] Found dataset: dataset_split/train
  [READY] Found dataset: dataset_split/test


In [8]:
# 7. Fast Smoke Test (~10 seconds)
# Verifies normalization, candidate blocking, classifier, decision layer & audit end-to-end
!python scripts/run_pipeline.py \
    --train-dir dataset_split/train \
    --test-dir dataset_split/test \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --smoke-test

   AMAZON ML CHALLENGE 2026 - COLAB & GPU HIGH-PERFORMANCE RUNNER
Run ID:            smoke_test_20260927_064755
Device Mode:       CUDA (NVIDIA L4)
CPU Workers:       12 threads (Configured n_jobs: -1)
System RAM:        37.36 GB free / 56.86 GB total
Train Dataset:     /content/amazon-ml-challenge/dataset_split/train
Test Dataset:      /content/amazon-ml-challenge/dataset_split/test
Run Output Dir:    /content/amazon-ml-challenge/output/runs/smoke_test_20260927_064755
Run Checkpoints:   /content/amazon-ml-challenge/checkpoints/runs/smoke_test_20260927_064755 (Resume: True)
Google Drive Sync: /content/drive/MyDrive/amazon-ml-challenge-2026 (runs/smoke_test_20260927_064755)
Smoke Test:        True
----------------------------------------------------------------------------

[Stage 1/3] Training Entity Resolution Pipeline...
  * Reading training ground truth from /content/amazon-ml-challenge/dataset_split/train/train_ground_truth.tsv...
  * Streaming training sources for 29 S1 entities (

In [9]:
# 9. Full Pipeline Execution: Training, Blocking & Chunked Inference (Crash-Resilient with Auto-Resume)
# Automatically uses GPU if available, throttles RAM safely, and checkpoints every chunk to Google Drive.
# Outputs are saved in dedicated run folders (output/runs/<run_id>/) so past runs are NEVER overwritten.
# If interrupted or timed out, re-running this cell resumes from the exact last completed batch!

# Sync latest code fixes from github repository
!git fetch origin main && git reset --hard origin/main

import os
import time

train_path = "data/shards/train" if os.path.exists("data/shards/train") else "dataset_split/train"
test_path = "data/shards/test" if os.path.exists("data/shards/test") else "dataset_split/test"

run_id = f"prod_calibrated_{time.strftime('%Y%m%d_%H%M%S')}"
print(f"Starting Run ID: {run_id}")
print(f"Train Dataset:   {train_path}")
print(f"Test Dataset:    {test_path}")

# Check if candidate_pairs already exists in Google Drive or previous run
cands_arg = ""
drive_cand_path = "/content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv"
if os.path.isfile(drive_cand_path):
    print(f"[*] Found pre-computed candidate_pairs.tsv in Google Drive: {drive_cand_path}")
    cands_arg = f"--candidate-pairs-path {drive_cand_path}"

!python scripts/run_pipeline.py \
    --train-dir $train_path \
    --test-dir $test_path \
    --run-id $run_id \
    --drive-sync-dir /content/drive/MyDrive/amazon-ml-challenge-2026 \
    --device auto \
    --batch-size 10000 \
    --n-jobs -1 \
    $cands_arg


Starting Run ID: prod_l4_fast_20260927_064759
Train Dataset:   data/shards/train
Test Dataset:    data/shards/test
[*] Found pre-computed candidate_pairs.tsv in Google Drive: /content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv
[*] Linking pre-computed candidates from /content/drive/MyDrive/amazon-ml-challenge-2026/output/candidate_pairs.tsv to /content/amazon-ml-challenge/output/runs/prod_l4_fast_20260927_064759/candidate_pairs.tsv...
   AMAZON ML CHALLENGE 2026 - COLAB & GPU HIGH-PERFORMANCE RUNNER
Run ID:            prod_l4_fast_20260927_064759
Device Mode:       CUDA (NVIDIA L4)
CPU Workers:       12 threads (Configured n_jobs: -1)
System RAM:        37.25 GB free / 56.86 GB total
Train Dataset:     /content/drive/MyDrive/amazon-ml-challenge-2026/data/shards/train
Test Dataset:      /content/drive/MyDrive/amazon-ml-challenge-2026/data/shards/test
Run Output Dir:    /content/amazon-ml-challenge/output/runs/prod_l4_fast_20260927_064759
Run Checkpoints:   /conten

In [10]:
# 10. Run Official Submission Validator on Generated Files
# Validates format, ID compliance, and required columns for competition submission (<50MB RAM, ~15s)
import os
test_path = "data/shards/test" if os.path.exists("data/shards/test") else "dataset_split/test"

!python utils/validate_submission.py \
    --matching output/latest/matching_results.tsv \
    --test-dir $test_path


ML Challenge 2026 — submission validator
  test dir: data/shards/test
  required S1 entities: 1732544
  matching_results.tsv: 1732544 rows (465905 empty, 1266639 non-empty).
  candidate_pairs.tsv: 1732544 rows (165959 empty, 1566585 non-empty).

PASS — no blocking issues found. Safe to submit.


In [11]:
# 11. Display Scorecard & Submission Preview Inline
import json
from pathlib import Path
from IPython.display import display, Markdown

scorecard_path = Path("output/latest/scorecard.json")
if scorecard_path.is_file():
    sc = json.loads(scorecard_path.read_text())
    md_table = f"""
### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **{sc.get('blocking_pair_recall', 0)*100:.2f}%** |
| | Space Reduction Rate | **{sc.get('candidate_reduction_rate', 0)*100:.2f}%** |
| | Candidates / Entity | **{sc.get('candidates_per_entity', 0):.1f}** |
| **2. GBDT Classifier** | Pair Precision | **{sc.get('classifier_pair_precision', 0)*100:.2f}%** |
| | Pair Recall | **{sc.get('classifier_pair_recall', 0)*100:.2f}%** |
| | Pairwise F1 | **{sc.get('classifier_pair_f1', 0):.4f}** |
| **3. Decision Layer** | **Official Macro F0.5** | **{sc.get('official_macro_f05', 0):.4f}** |
| | Singleton Sub-Score | **{sc.get('singleton_macro_f05', 0):.4f}** |
| | Matched Sub-Score | **{sc.get('matched_macro_f05', 0):.4f}** |
"""
    display(Markdown(md_table))

print("\n--- Latest Run Output Files ---")
!ls -lh output/latest/
print("\n--- First 5 rows of matching_results.tsv ---")
!head -n 6 output/latest/matching_results.tsv



### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **100.00%** |
| | Space Reduction Rate | **95.52%** |
| | Candidates / Entity | **4.3** |
| **2. GBDT Classifier** | Pair Precision | **100.00%** |
| | Pair Recall | **98.61%** |
| | Pairwise F1 | **0.9930** |
| **3. Decision Layer** | **Official Macro F0.5** | **0.9976** |
| | Singleton Sub-Score | **1.0000** |
| | Matched Sub-Score | **0.9975** |



--- Latest Run Output Files ---
total 780M
-rw------- 1 root root 711M Sep 26 17:44 candidate_pairs.tsv
-rw-r--r-- 1 root root  69M Sep 27 10:33 matching_results.tsv
-rw-r--r-- 1 root root  721 Sep 27 06:47 scorecard.json

--- First 5 rows of matching_results.tsv ---
source1_entity_id	matched_entity_ids
S1-100000556	S2-773905311,S3-502189990,S3-912715604,S3-972261068
S1-100001013	S2-135727199,S2-176472141,S3-460176037,S3-969494322
S1-100001295	S2-334300764,S3-327023247,S3-380201812,S3-409832146,S3-830159204
S1-100001834	S2-335470090,S2-900957143,S3-233043065,S3-420734787
S1-100003037	


In [12]:
# 11. Display Scorecard & Submission Preview Inline
import json
from pathlib import Path
from IPython.display import display, Markdown

scorecard_path = Path("output/scorecard.json")
if scorecard_path.is_file():
    sc = json.loads(scorecard_path.read_text())
    md_table = f"""
### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **{sc.get('blocking_pair_recall', 0)*100:.2f}%** |
| | Space Reduction Rate | **{sc.get('candidate_reduction_rate', 0)*100:.2f}%** |
| | Candidates / Entity | **{sc.get('candidates_per_entity', 0):.1f}** |
| **2. GBDT Classifier** | Pair Precision | **{sc.get('classifier_pair_precision', 0)*100:.2f}%** |
| | Pair Recall | **{sc.get('classifier_pair_recall', 0)*100:.2f}%** |
| | Pairwise F1 | **{sc.get('classifier_pair_f1', 0):.4f}** |
| **3. Decision Layer** | **Official Macro F0.5** | **{sc.get('official_macro_f05', 0):.4f}** |
| | Singleton Sub-Score | **{sc.get('singleton_macro_f05', 0):.4f}** |
| | Matched Sub-Score | **{sc.get('matched_macro_f05', 0):.4f}** |
"""
    display(Markdown(md_table))

print("\n--- First 5 rows of candidate_pairs.tsv ---")
!head -n 6 output/candidate_pairs.tsv
print("\n--- First 5 rows of matching_results.tsv ---")
!head -n 6 output/matching_results.tsv


### 📊 Evaluation Scorecard (3 Pipeline Tiers)
| Pipeline Tier | Metric | Value |
| :--- | :--- | :--- |
| **1. Candidate Blocking** | Pair Recall | **100.00%** |
| | Space Reduction Rate | **95.52%** |
| | Candidates / Entity | **4.3** |
| **2. GBDT Classifier** | Pair Precision | **100.00%** |
| | Pair Recall | **98.61%** |
| | Pairwise F1 | **0.9930** |
| **3. Decision Layer** | **Official Macro F0.5** | **0.9976** |
| | Singleton Sub-Score | **1.0000** |
| | Matched Sub-Score | **0.9975** |



--- First 5 rows of candidate_pairs.tsv ---
source1_entity_id	candidate_entity_ids
S1-100000556	S2-773905311,S3-502189990,S3-912715604,S3-972261068
S1-100001013	S2-135727199,S2-176472141,S3-460176037,S3-969494322
S1-100001295	S2-334300764,S2-964627294,S3-327023247,S3-380201812,S3-409832146,S3-595174672,S3-830159204
S1-100001834	S2-113790003,S2-159173227,S2-167964865,S2-185863552,S2-202408762,S2-232276569,S2-245346859,S2-264836656,S2-31854181,S2-335470090,S2-35634267,S2-405951741,S2-420890349,S2-510110075,S2-512701337,S2-534787934,S2-557008499,S2-592911711,S2-614965037,S2-619122220,S2-759924505,S2-789668698,S2-800040544,S2-810645020,S2-899874979,S2-900957143,S2-933146793,S2-974920471,S3-139890156,S3-156917626,S3-197996556,S3-202655251,S3-233043065,S3-242683066,S3-25018433,S3-258407774,S3-28236837,S3-307349108,S3-390794811,S3-420734787,S3-436567976,S3-607060770,S3-610419748,S3-650485403,S3-657069047,S3-662629015,S3-681574841,S3-79964484,S3-882426509,S3-913381011
S1-100003037	

--- First